<a href="https://colab.research.google.com/github/rix031110/LLM_implementation/blob/MAria/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 1. Setup & Dependencies

In [2]:
# Install required packages
!pip install -q transformers datasets faiss-cpu sentence-transformers torch numpy pandas matplotlib seaborn scikit-learn pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 18.7 MB/s eta 0:00:00


In [45]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
import time
import warnings
import os
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModel,
    GPT2LMHeadModel,
    GPT2Tokenizer,
    T5Tokenizer,
    T5ForConditionalGeneration,
    set_seed
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import faiss
from sklearn.metrics.pairwise import cosine_similarity
from pypdf import PdfReader

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
# Define the path to your uploaded PDF file
pdf_file_path = '/content/drive/MyDrive/LLM/ENG_REC.pdf'

# Check if the file exists
if not os.path.exists(pdf_file_path):
    print(f"Error: The file '{pdf_file_path}' was not found. Please ensure you have uploaded it correctly.")
else:
    # Create a PdfReader object
    reader = PdfReader(pdf_file_path)

    # Get the number of pages
    num_pages = len(reader.pages)
    print(f"The PDF has {num_pages} page(s).")

    # Extract text from each page
    extracted_text = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            extracted_text.append(f"--- Page {i+1} ---\n{text}")
        else:
            extracted_text.append(f"--- Page {i+1} ---\n[No text found on this page]")

    # Print the first 500 characters of the extracted text
    full_text = "\n".join(extracted_text)
    print("\n--- Extracted Text (first 500 chars) ---")
    print(full_text[:800])

    # You can also store the full text in a variable for further processing
    # For example, to integrate with a RAG pipeline:
    # document_for_rag = {'id': 'pdf_doc_1', 'title': 'Sample PDF Content', 'text': full_text}
    # print(document_for_rag)

The PDF has 126 page(s).

--- Extracted Text (first 500 chars) ---
--- Page 1 ---
INTERNATIONAL STANDARDS  
ON COMBATING MONEY LAUNDERING  
AND THE FINANCING OF  
TERRORISM & PROLIFERATION
The FATF Recommendations
February 2012

--- Page 2 ---
 
 
  
FINANCIAL ACTION TASK FORCE 
 
The Financial Action Task Force (FATF) is an independent inter -governmental body that develops and 
promotes policies to protect the global financial system against money laundering, terrorist financing  
and the financing of proliferation of weapons of mass destruction .  The FATF Recommendations are 
recognised as the global anti-money laundering (AML) and counter-terrorist financing (CFT) standard. 
For more information about the FATF, please visit the website:  
www.fatf-gafi.org 
 
© 2012 FATF/OECD. All rights reserved. 
No reproduction or translation of this publication m


---
## 3. Building the Knowledge Base

First, we need a collection of documents to retrieve from. We'll use a sample dataset and create a simple knowledge base.

In [22]:
# basic_knowledge creates 1 document per page
'''basic_knowledge = []

# Split full_text into individual pages
page_contents = full_text.split('--- Page ')[1:]  # Skip the first empty split

for i, page_content in enumerate(page_contents):
    page_number_str, text_content = page_content.split(' ---\n', 1)
    page_number = int(page_number_str.strip())

    basic_knowledge.append({
        'id': page_number,
        'title': f'FAFT reccomendations {page_number}',
        'text': text_content.strip()
    })

# You can print the first few entries to verify
print(basic_knowledge[100]) # if there's a second page'''

{'id': 101, 'title': 'FAFT reccomendations 101', 'text': 'THE FATF RECOMMENDATIONS \nINTERNATIONAL STANDARDS ON COMBATING MONEY LAUNDERING AND THE FINANCING OF TERRORISM & PROLIFERATION \n\uf0e3 2012 OECD/FATF  99 \n(c)  Oral declaration system for all travellers: In this system, all travellers are required to \norally decl are if they carry an amount of currency or BNIs above a prescribed \nthreshold. Usually, this is done at customs entry points by requiring travellers to \nchoose between the “red channel” (goods to declare) and the “green channel” (nothing \nto declare). The choice of channel that the traveller makes is considered to be the oral \ndeclaration. In practice, travellers do not declare in writing, but are required to \nactively report to a customs official.  \nDisclosure system: \n4.  Countries may opt for a system whereby t ravellers are required to provide the authorities \nwith appropriate information upon request. In such systems, there is no requirement for \ntrave

In [82]:
# basic_knowledge creates 1 document per paragraph (max 300 words, no paragraph overlap)
basic_knowledge = []

# Split full_text into individual pages
page_contents = full_text.split('--- Page ')[1:]  # Skip the first empty split

CHUNK_SIZE = 300  # maximum number of words per chunk
chunk_id = 0

for page_content in page_contents:
    page_number_str, text_content = page_content.split(' ---\n', 1)
    page_number = int(page_number_str.strip())
    text_content = text_content.strip()

    # Split into paragraphs (by line breaks)
    paragraphs = [p.strip() for p in text_content.split('\n') if p.strip()]

    current_chunk = []
    current_count = 0

    for para in paragraphs:
        words = para.split()

        # If a single paragraph already exceeds the limit, split it into 300-word blocks
        if len(words) > CHUNK_SIZE:
            # First, flush the accumulated chunk if there is one
            if current_chunk:
                chunk_text = ' '.join(current_chunk)
                basic_knowledge.append({
                    'id': chunk_id,
                    'title': f'FATF recommendations (page {page_number})',
                    'text': chunk_text
                })
                chunk_id += 1
                current_chunk = []
                current_count = 0

            # Split the long paragraph
            for i in range(0, len(words), CHUNK_SIZE):
                sub = words[i:i + CHUNK_SIZE]
                basic_knowledge.append({
                    'id': chunk_id,
                    'title': f'FATF recommendations (page {page_number})',
                    'text': ' '.join(sub)
                })
                chunk_id += 1
            continue

        # If adding this paragraph exceeds the limit, flush the current chunk and start a new one
        if current_count + len(words) > CHUNK_SIZE:
            chunk_text = ' '.join(current_chunk)
            basic_knowledge.append({
                'id': chunk_id,
                'title': f'FATF recommendations (page {page_number})',
                'text': chunk_text
            })
            chunk_id += 1
            current_chunk = words
            current_count = len(words)
        else:
            current_chunk.extend(words)
            current_count += len(words)

    # Save any remaining content when the page ends
    if current_chunk:
        chunk_text = ' '.join(current_chunk)
        basic_knowledge.append({
            'id': chunk_id,
            'title': f'FATF recommendations (page {page_number})',
            'text': chunk_text
        })
        chunk_id += 1

print(f"Total chunks created: {len(basic_knowledge)}")
print("\nExample chunk:")
print(basic_knowledge[0])

Total chunks created: 211

Example chunk:
{'id': 0, 'title': 'FATF recommendations (page 1)', 'text': 'INTERNATIONAL STANDARDS ON COMBATING MONEY LAUNDERING AND THE FINANCING OF TERRORISM & PROLIFERATION The FATF Recommendations February 2012'}


In [83]:
df_kb = pd.DataFrame(basic_knowledge)
print(f"Knowledge Base: {len(df_kb)} documents\n")
print(df_kb[['id', 'title']].head(10).to_string(index=False))

Knowledge Base: 211 documents

 id                         title
  0 FATF recommendations (page 1)
  1 FATF recommendations (page 2)
  2 FATF recommendations (page 3)
  3 FATF recommendations (page 4)
  4 FATF recommendations (page 5)
  5 FATF recommendations (page 6)
  6 FATF recommendations (page 7)
  7 FATF recommendations (page 8)
  8 FATF recommendations (page 9)
  9 FATF recommendations (page 9)


---
## 4. Step 1: RETRIEVAL - Building the Vector Database

To retrieve relevant documents, we need to:
1. Convert documents to **embeddings** (dense vectors)
2. Store embeddings in a **vector database**
3. Convert queries to embeddings
4. Find most similar documents using **semantic search**

### 4.1 Create Document Embeddings

We'll use **Sentence-BERT** to create high-quality embeddings.

In [84]:
# Load embedding model (Sentence-BERT)
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and efficient
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding model loaded. Dimension: {embedding_dim}")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded. Dimension: 384


In [85]:
# Generate embeddings for all documents
print("\nGenerating embeddings for documents...")
start_time = time.time()

# Combine title and text for richer embeddings
df_kb['combined_text'] = df_kb['title'] + ': ' + df_kb['text']
documents = df_kb['combined_text'].tolist()

# Encode all documents
doc_embeddings = embedding_model.encode(documents, show_progress_bar=True)
doc_embeddings = np.array(doc_embeddings).astype('float32')

embedding_time = time.time() - start_time
print(f"\nEmbeddings generated in {embedding_time:.2f} seconds")
print(f"Embeddings shape: {doc_embeddings.shape}")


Generating embeddings for documents...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Embeddings generated in 30.74 seconds
Embeddings shape: (211, 384)


### 4.2 Build FAISS Index

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search in high-dimensional spaces.

In [86]:
# Create FAISS index
print("Building FAISS index...")
index = faiss.IndexFlatL2(embedding_dim)  # L2 distance (Euclidean)

# Normalize vectors for cosine similarity
faiss.normalize_L2(doc_embeddings)
index.add(doc_embeddings)

print(f"FAISS index built with {index.ntotal} vectors")

Building FAISS index...
FAISS index built with 211 vectors


### 4.3 Implement Retrieval Function

In [87]:
def retrieve_documents(query: str, k: int = 3) -> List[Dict]:
    """
    Retrieve top-k most relevant documents for a query.

    Args:
        query: The search query
        k: Number of documents to retrieve

    Returns:
        List of retrieved documents with scores
    """
    # Encode query
    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    faiss.normalize_L2(query_embedding)

    # Search in FAISS index
    distances, indices = index.search(query_embedding, k)

    #distances: A 2D array of shape (1, k) — the similarity/distance scores for each returned neighbor.
    #indices: A 2D array of shape (1, k) — the positions of the nearest neighbors in the original index.

    # Convert distances to similarity scores (cosine similarity)
    similarities = 1 - distances[0]

    # Retrieve documents
    results = []
    for idx, score in zip(indices[0], similarities):
        doc = df_kb.iloc[idx].to_dict()
        doc['retrieval_score'] = float(score)
        results.append(doc)

    return results

### 4.4 Test Retrieval

In [88]:
# Test queries
test_queries = [
    "What is FATF?"
]

print("Testing Retrieval System:\n")
print("="*80)

for query in test_queries:
    print(f"\nQuery: '{query}'\n")
    retrieved = retrieve_documents(query, k=3)

    for i, doc in enumerate(retrieved, 1):
        print(f"{i}. {doc['title']} (score: {doc['retrieval_score']:.3f})")
        print(f"   {doc['text'][:100]}...\n")
    print("="*80)

Testing Retrieval System:


Query: 'What is FATF?'

1. FATF recommendations (page 66) (score: 0.408)
    A pension, superannuation or similar scheme that provides retirement benefits to employees, where ...

2. FATF recommendations (page 11) (score: 0.182)
   THE FATF RECOMMENDATIONS INTERNATIONAL STANDARDS ON COMBATING MONEY LAUNDERING AND THE FINANCING OF ...

3. FATF recommendations (page 2) (score: 0.162)
   FINANCIAL ACTION TASK FORCE The Financial Action Task Force (FATF) is an independent inter -governme...



---
## 5. Step 2: AUGMENTATION - Building the Prompt

Now we combine the query with retrieved context to create an augmented prompt.

In [89]:
def create_augmented_prompt(query: str, retrieved_docs: List[Dict], max_context_length: int = 900) -> str:
    """
    Create an augmented prompt by combining query with retrieved context.

    Args:
        query: User's question
        retrieved_docs: List of retrieved documents
        max_context_length: Maximum characters for context

    Returns:
        Augmented prompt string
    """
    # Build context from retrieved documents
    context_parts = []
    current_length = 0

    for doc in retrieved_docs:
        doc_text = f"[{doc['title']}] {doc['text']}"
        if current_length + len(doc_text) <= max_context_length:
            context_parts.append(doc_text)
            current_length += len(doc_text)
        else:
            break

    context = "\n\n".join(context_parts)

    # Create prompt template
    #prompt = f"""Context:
    #{context}

    #Question: {query}

    #Answer based on the context above:"""

    # Create prompt template
    prompt = f"""Answer the question using only the context below.

    Context: {context}
    Question: {query}
    Answer:"""

    return prompt

In [90]:
# Example augmented prompt
query = "when did FATF Recommendations revised for the first time?"
retrieved = retrieve_documents(query, k=2)
augmented_prompt = create_augmented_prompt(query, retrieved)

print("Example Augmented Prompt:\n")
print("="*80)
print(augmented_prompt)
print("="*80)

Example Augmented Prompt:

Answer the question using only the context below.

    Context: [FATF recommendations (page 89)] THE FATF RECOMMENDATIONS INTERNATIONAL STANDARDS ON COMBATING MONEY LAUNDERING AND THE FINANCING OF TERRORISM & PROLIFERATION  2012 OECD/FATF 87 and beneficial ownership information or requests for assistance in locating beneficial owners residing abroad.

[FATF recommendations (page 23)] THE FATF RECOMMENDATIONS INTERNATIONAL STANDARDS ON COMBATING MONEY LAUNDERING AND THE FINANCING OF TERRORISM & PROLIFERATION  2012 OECD/FATF 21 (c) Trust and company service providers should be required to report suspicious transactions for a client when, on behalf of or for a client, they engage in a transaction in relation to the activities referred to in paragraph (e) of Recommendation 22.
    Question: when did FATF Recommendations revised for the first time?
    Answer:


---
## 6. Step 3: GENERATION - Producing the Answer

Now we feed the augmented prompt to a language model to generate an answer.

In [46]:
# Load generation model (GPT-2)
#print("Loading generation model...")
#gen_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
#gen_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
#gen_tokenizer.pad_token = gen_tokenizer.eos_token
#print("Generation model loaded (GPT-2)")


print("Loading generation model...")
gen_tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')
gen_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base').to(device)
print("Generation model loaded (Flan-T5-base)")

Loading generation model...


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generation model loaded (Flan-T5-base)


In [39]:
#generate answer con GPT2
'''def generate_answer(prompt: str, max_length: int = 500) -> str:
    """
    Generate an answer using the language model.

    Args:
        prompt: The augmented prompt with context
        max_length: Maximum length of generated answer

    Returns:
        Generated answer string
    """
    input_ids = gen_tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Generate with controlled parameters
    output = gen_model.generate(
        input_ids,
        max_length=input_ids.shape[1] + max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        num_return_sequences=1,
        pad_token_id=gen_tokenizer.eos_token_id
    )

    # Decode and extract only the new generated part
    full_text = gen_tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract answer (everything after the prompt)
    if "Answer based on the context above:" in full_text:
        answer = full_text.split("Answer based on the context above:")[-1].strip()
    else:
        answer = full_text[len(prompt):].strip()

    return answer'''

In [91]:
#generate answer con Flan-T5
def generate_answer(prompt: str, max_new_tokens: int = 200) -> str:
    """
    Generate an answer using Flan-T5 (seq2seq).
    """
    # Flan-T5 soporta hasta 512 tokens de entrada
    input_ids = gen_tokenizer.encode(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512
    ).to(device)

    output = gen_model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        num_return_sequences=1
    )

    # En seq2seq, la salida es SOLO la respuesta (no incluye el prompt)
    answer = gen_tokenizer.decode(output[0], skip_special_tokens=True)
    return answer.strip()

---
## 7. Complete RAG Pipeline

In [92]:
def rag_pipeline(query: str, k: int = 3, verbose: bool = True) -> Dict:
    """
    Complete RAG pipeline: Retrieve → Augment → Generate

    Args:
        query: User's question
        k: Number of documents to retrieve
        verbose: Print intermediate steps

    Returns:
        Dictionary with answer, retrieved docs, and metadata
    """
    start_time = time.time()

    # Step 1: Retrieve
    if verbose:
        print(f"Query: '{query}'\n")
        print("[1/3] Retrieving relevant documents...")

    retrieval_start = time.time()
    retrieved_docs = retrieve_documents(query, k=k)
    retrieval_time = time.time() - retrieval_start

    if verbose:
        print(f"Retrieved {len(retrieved_docs)} documents in {retrieval_time:.3f}s")
        for i, doc in enumerate(retrieved_docs, 1):
            print(f"  {i}. {doc['title']} (relevance: {doc['retrieval_score']:.3f})")

    # Step 2: Augment
    if verbose:
        print("\n[2/3] Creating augmented prompt...")

    augment_start = time.time()
    augmented_prompt = create_augmented_prompt(query, retrieved_docs)
    augment_time = time.time() - augment_start

    if verbose:
        print(f"Augmentation completed in {augment_time:.3f}s")

    # Step 3: Generate
    if verbose:
        print("\n[3/3] Generating answer...")

    gen_start = time.time()
    answer = generate_answer(augmented_prompt)
    gen_time = time.time() - gen_start

    if verbose:
        print(f"Answer generated in {gen_time:.3f}s\n")

    total_time = time.time() - start_time

    return {
        'query': query,
        'answer': answer,
        'retrieved_docs': retrieved_docs,
        'augmented_prompt': augmented_prompt,
        'timings': {
            'retrieval': retrieval_time,
            'augmentation': augment_time,
            'generation': gen_time,
            'total': total_time
        }
    }

---
## 8. Test the Complete RAG Pipeline

In [98]:
# Example queries to test the RAG pipeline
test_queries = [
    "On which year were the recommendations last updated?"
]

print("\n" + "="*80)
print("RAG PIPELINE DEMO")
print("="*80)

for query in test_queries[:1]:  # Test with first query
    print(f"\n\nProcessing: {query}\n")
    result = rag_pipeline(query, k=2, verbose=True)

    print("\n" + "-"*80)
    print("FINAL ANSWER:")
    print("-"*80)
    print(result['answer'])
    print("-"*80)
    print(f"\nTotal time: {result['timings']['total']:.3f} seconds")


RAG PIPELINE DEMO


Processing: On which year were the recommendations last updated?

Query: 'On which year were the recommendations last updated?'

[1/3] Retrieving relevant documents...
Retrieved 2 documents in 0.024s
  1. FATF recommendations (page 9) (relevance: -0.448)
  2. FATF recommendations (page 50) (relevance: -0.472)

[2/3] Creating augmented prompt...
Augmentation completed in 0.000s

[3/3] Generating answer...
Answer generated in 1.697s


--------------------------------------------------------------------------------
FINAL ANSWER:
--------------------------------------------------------------------------------
2003
--------------------------------------------------------------------------------

Total time: 1.722 seconds
